# Alpaca 模拟交易教程 — 从回测到实盘的完整闭环

本教程演示 open-xquant 从回测到模拟交易的完整工作流：

1. **FillPriceMode** — 不同成交价模式的压力测试
2. **OrderGenerator** — 收盘后生成次日交易计划，供交易员审核
3. **Engine.setup() + step()** — 逐 bar 执行，实盘模式的基础
4. **ExecutionReport** — 回测 vs 实盘成交偏差分析
5. **Alpaca 集成** — 从 Alpaca 获取真实行情 + 下单（需要 API Key）

**前 4 节不需要 Alpaca API Key**，使用本地构造数据即可运行。第 5 节需要设置环境变量：

```bash
pip install open-xquant[live]
export ALPACA_API_KEY="your-paper-api-key"
export ALPACA_SECRET_KEY="your-paper-secret-key"
```

## 准备工作：构造本地行情 + 策略

为了让教程独立运行（不依赖外部数据源），我们构造一段模拟行情数据：下跌 → 上涨 → 下跌，触发 SMA 金叉/死叉信号。

In [1]:
import pandas as pd
from decimal import Decimal

from oxq.core import Engine, Strategy
from oxq.core.types import Fill, Order, Position, Portfolio
from oxq.indicators import SMA
from oxq.portfolio import ExecutionReport
from oxq.rules import EntryRule, ExitRule
from oxq.signals import Crossover
from oxq.trade import FillPriceMode, SimBroker, generate_orders
from oxq.universe import StaticUniverse


def make_sample_data() -> dict[str, pd.DataFrame]:
    """构造模拟行情：下跌 → 上涨 → 下跌（120 bars）。"""
    n = 120
    dates = pd.bdate_range("2024-01-01", periods=n)
    closes: list[float] = []
    for i in range(50):
        closes.append(200 - i * 2)   # 200 → 102
    for i in range(40):
        closes.append(102 + i * 2)   # 102 → 180
    for i in range(30):
        closes.append(180 - i * 2)   # 180 → 122
    return {
        "AAPL": pd.DataFrame({
            "open": [c - 1 for c in closes],
            "high": [c + 2 for c in closes],
            "low": [c - 2 for c in closes],
            "close": closes,
            "volume": [1_000_000] * n,
        }, index=dates),
    }


class FakeMarketDataProvider:
    """本地行情 Provider，用于教程演示。"""
    def __init__(self, data: dict[str, pd.DataFrame]) -> None:
        self._data = data
    def get_bars(self, symbol: str, start: str, end: str) -> pd.DataFrame:
        df = self._data[symbol]
        return df[(df.index >= start) & (df.index <= end)]
    def get_latest(self, symbol: str) -> pd.Series:
        return self._data[symbol].iloc[-1]


strategy = Strategy(
    name="sma_crossover",
    hypothesis="SMA10 上穿 SMA50 时买入，下穿时卖出",
    universe=StaticUniverse(("AAPL",)),
    indicators={
        "sma_10": (SMA(), {"period": 10}),
        "sma_50": (SMA(), {"period": 50}),
    },
    signals={
        "golden_cross": (Crossover(), {"fast": "sma_10", "slow": "sma_50"}),
    },
    entry_rules=[EntryRule(signal="golden_cross", shares=100)],
    exit_rules=[ExitRule(fast="sma_10", slow="sma_50")],
)

data = make_sample_data()
market = FakeMarketDataProvider(data)

print(f"行情数据: AAPL {len(data['AAPL'])} bars")
print(f"日期范围: {data['AAPL'].index[0].date()} ~ {data['AAPL'].index[-1].date()}")
print(f"策略: {strategy.name}")

行情数据: AAPL 120 bars
日期范围: 2024-01-01 ~ 2024-06-14
策略: sma_crossover


---
## 1. FillPriceMode — 不同成交价模式的压力测试

回测中，订单「以什么价格成交」对结果影响很大。`SimBroker` 支持 4 种成交价模式：

| 模式 | 含义 | 适用场景 |
|------|------|----------|
| `CLOSE` | 当前 bar 收盘价（默认） | 标准回测 |
| `NEXT_OPEN` | 次日开盘价 | 最接近真实执行（信号收盘后产生，次日开盘执行） |
| `NEXT_HIGH` | 次日最高价 | 压力测试：买在最高（最差买入） |
| `NEXT_LOW` | 次日最低价 | 压力测试：卖在最低（最差卖出） |

用同一策略、同一数据，分别用 4 种模式回测，观察收益差异：

In [2]:
modes = {
    "CLOSE (默认)": FillPriceMode.CLOSE,
    "NEXT_OPEN": FillPriceMode.NEXT_OPEN,
    "NEXT_HIGH (买最差)": FillPriceMode.NEXT_HIGH,
    "NEXT_LOW (卖最差)": FillPriceMode.NEXT_LOW,
}

results_by_mode = {}
for label, mode in modes.items():
    broker = SimBroker(fill_price_mode=mode)
    result = Engine().run(
        strategy, market=market, broker=broker,
        start="2024-01-01", end="2024-12-31",
    )
    results_by_mode[label] = result

print(f"{'模式':<20} {'总收益':>8} {'Sharpe':>8} {'最大回撤':>8} {'交易数':>6}")
print("-" * 54)
for label, r in results_by_mode.items():
    print(f"{label:<20} {r.total_return():>7.2%} {r.sharpe_ratio():>8.2f} {r.max_drawdown():>7.2%} {len(r.trades):>6}")

模式                        总收益   Sharpe     最大回撤    交易数
------------------------------------------------------
CLOSE (默认)             0.60%     0.72  -3.45%      2
NEXT_OPEN              0.20%     0.23  -3.75%      2
NEXT_HIGH (买最差)        0.20%     0.23  -3.47%      2
NEXT_LOW (卖最差)         0.20%     0.23  -3.84%      2


可以看到不同成交价假设下，收益差异明显。`CLOSE` 模式是「理想化」的——信号产生的那个 bar 就以收盘价成交，现实中做不到。`NEXT_OPEN` 更接近真实场景：收盘后跑策略产生信号，次日开盘下单。

**最佳实践**：先用 `CLOSE` 模式快速验证策略逻辑，然后用 `NEXT_OPEN` 评估真实可行性，最后用 `NEXT_HIGH`/`NEXT_LOW` 做压力测试。

我们来看看各模式下的具体成交价差异：

In [3]:
# 对比各模式下第一笔买入的成交价
print("各模式下第一笔买入成交价：")
for label, r in results_by_mode.items():
    if r.trades:
        t = r.trades[0]
        print(f"  {label:<20} {t.order.side} {t.order.shares} 股 @ ${t.filled_price}")

各模式下第一笔买入成交价：
  CLOSE (默认)           BUY 100 股 @ $138.0
  NEXT_OPEN            BUY 100 股 @ $139.0
  NEXT_HIGH (买最差)      BUY 100 股 @ $142.0
  NEXT_LOW (卖最差)       BUY 100 股 @ $138.0


---
## 2. OrderGenerator — 收盘后生成次日交易计划

`OrderGenerator` 是一个独立的纯函数工具，用于「收盘后跑策略，生成次日交易计划」的场景。它与引擎内部的 `RebalanceRule` 功能类似，但职责不同：

| | RebalanceRule | OrderGenerator |
|---|---|---|
| 运行位置 | Engine 内部循环 | Engine 外部 |
| 输出 | 直接提交 Order | `PlannedOrder`（带上下文） |
| 人工审核 | 不支持 | 支持（核心用途） |

典型场景：收盘后运行策略得到目标权重，`generate_orders()` 将目标权重转化为具体订单列表，交易员审核后次日执行。

In [4]:
# 场景：当前持有 AAPL 30 股，策略建议调仓为 AAPL 40% / GOOG 30% / MSFT 30%
positions = {
    "AAPL": Position(symbol="AAPL", shares=30, avg_cost=Decimal("150")),
}
target_weights = {
    "AAPL": Decimal("0.4"),
    "GOOG": Decimal("0.3"),
    "MSFT": Decimal("0.3"),
}
prices = {
    "AAPL": Decimal("180"),
    "GOOG": Decimal("140"),
    "MSFT": Decimal("420"),
}
total_capital = Decimal("100000")

planned = generate_orders(
    target_weights=target_weights,
    positions=positions,
    prices=prices,
    total_capital=total_capital,
)

print(f"总资金: ${total_capital:,}  |  当前持仓: AAPL x {positions['AAPL'].shares}")
print()
print(f"{'Symbol':<8} {'Side':>5} {'Shares':>7} {'当前':>6} {'目标':>6} {'当前权重':>8} {'目标权重':>8} {'预估金额':>10}")
print("-" * 72)
for p in planned:
    print(
        f"{p.order.symbol:<8} {p.order.side:>5} {p.order.shares:>7} "
        f"{p.current_shares:>6} {p.target_shares:>6} "
        f"{p.current_weight:>7.1%} {p.target_weight:>7.1%} "
        f"${p.estimated_amount:>8,}"
    )
print(f"\n共 {len(planned)} 笔订单待审核，全部为 market 单（次日开盘执行）")

总资金: $100,000  |  当前持仓: AAPL x 30

Symbol    Side  Shares     当前     目标     当前权重     目标权重       预估金额
------------------------------------------------------------------------
AAPL       BUY     192     30    222    5.4%   40.0% $  34,560
GOOG       BUY     214      0    214    0.0%   30.0% $  29,960
MSFT       BUY      71      0     71    0.0%   30.0% $  29,820

共 3 笔订单待审核，全部为 market 单（次日开盘执行）


`PlannedOrder` 不仅包含 `Order` 对象，还带有上下文信息（当前持仓、目标持仓、权重、预估金额），方便交易员做出判断。

### A 股 lot_size 支持

A 股最小交易单位是 100 股（一手），`generate_orders()` 通过 `lot_size` 参数支持整手约束：

In [5]:
# A 股场景：lot_size=100
a_share_planned = generate_orders(
    target_weights={"600519": Decimal("0.5")},
    positions={},
    prices={"600519": Decimal("1800")},
    total_capital=Decimal("1000000"),
    lot_size=100,
)

for p in a_share_planned:
    print(f"贵州茅台: BUY {p.order.shares} 股 (整手)")
    print(f"  计算过程: floor(1000000 * 0.5 / 1800 / 100) * 100 = {p.order.shares}")

贵州茅台: BUY 200 股 (整手)
  计算过程: floor(1000000 * 0.5 / 1800 / 100) * 100 = 200


---
## 3. Engine.setup() + step() — 逐 bar 执行

`Engine.run()` 一次性跑完全部 bar，适合回测。但实盘中，每天（或每分钟）只有一根新 bar，需要逐步执行。

`Engine` 提供了 `setup()` + `step()` 接口：

```
setup()   → 加载数据、计算指标/信号（一次性）
step(date) → 处理单根 bar（风控 → 挂单 → 规则 → 撮合 → 记录）
```

外部调度器（cron / APScheduler / 手动脚本）控制何时调用 `step()`。Engine 本身不含定时器。

In [6]:
# 方式 A: 传统 run() 一次性执行
engine_a = Engine()
result_a = engine_a.run(
    strategy, market=market, broker=SimBroker(),
    start="2024-01-01", end="2024-12-31",
)

# 方式 B: setup() + step() 逐 bar 执行
engine_b = Engine()
engine_b.setup(
    strategy=strategy, market=market, broker=SimBroker(),
    start="2024-01-01", end="2024-12-31",
)

print(f"共 {len(engine_b.dates)} 根 bar，逐 bar 执行中...")
for i, date in enumerate(engine_b.dates):
    engine_b.step(date)
    # 实盘中，这里可以加入日志、通知、风控检查
    if (i + 1) % 40 == 0:
        eq = engine_b.result.equity_curve[-1][1]
        print(f"  [{date.date()}] bar {i+1}/{len(engine_b.dates)}, equity=${eq:,.0f}")

result_b = engine_b.result
print(f"\nrun()  总收益: {result_a.total_return():.2%},  trades: {len(result_a.trades)}")
print(f"step() 总收益: {result_b.total_return():.2%},  trades: {len(result_b.trades)}")

# 验证两种方式结果完全一致
eq_a = [v for _, v in result_a.equity_curve]
eq_b = [v for _, v in result_b.equity_curve]
assert all(abs(a - b) < 0.01 for a, b in zip(eq_a, eq_b))
print("\n结果完全一致 ✓")

共 120 根 bar，逐 bar 执行中...
  [2024-02-23] bar 40/120, equity=$100,000
  [2024-04-19] bar 80/120, equity=$102,200
  [2024-06-14] bar 120/120, equity=$100,600

run()  总收益: 0.60%,  trades: 2
step() 总收益: 0.60%,  trades: 2

结果完全一致 ✓


---
## 4. ExecutionReport — 回测 vs 实盘成交偏差分析

实盘交易后，需要对比「回测预期成交」和「实际成交」的差异。`ExecutionReport` 按 (symbol, date, side) 聚合成交记录，做 full outer join，计算滑点和手续费偏差。

典型场景：
1. 用 `CLOSE` 模式回测得到 sim_fills
2. 用 `NEXT_OPEN` 模式模拟实盘得到 live_fills（更接近真实）
3. `ExecutionReport` 对比两组成交，量化偏差

| 指标 | 含义 |
|------|------|
| `price_slippage` | (实际价 - 预期价) / 预期价 |
| `shares_diff` | 实际股数 - 预期股数 |
| `fee_diff` | 实际手续费 - 预期手续费 |

In [7]:
# 用 CLOSE 模式作为「回测预期」，NEXT_OPEN 作为「模拟实盘」
sim_result = results_by_mode["CLOSE (默认)"]
live_result = results_by_mode["NEXT_OPEN"]

report = ExecutionReport(
    sim_fills=sim_result.trades,
    live_fills=live_result.trades,
)

# 逐笔对比
print(f"{'Symbol':<6} {'Date':<12} {'Side':<5} {'Sim价':>8} {'Live价':>8} {'滑点':>8} {'股数差':>6}")
print("-" * 60)
for c in report.comparisons:
    print(
        f"{c.symbol:<6} {c.date:<12} {c.side:<5} "
        f"${c.sim_avg_price:>7.2f} ${c.live_avg_price:>7.2f} "
        f"{c.price_slippage:>7.2%} {c.shares_diff:>6}"
    )

# 汇总统计
s = report.summary()
print("\n汇总:")
print(f"  总交易笔数:   {s['total_trades']}")
print(f"  匹配交易:     {s['matched_trades']}")
print(f"  仅回测有:     {s['sim_only_trades']}")
print(f"  仅实盘有:     {s['live_only_trades']}")
print(f"  平均滑点:     {s['avg_price_slippage']:.4%}")
print(f"  手续费偏差:   ${s['total_fee_diff']}")

Symbol Date         Side      Sim价    Live价       滑点    股数差
------------------------------------------------------------
AAPL   2024-04-04   BUY   $ 138.00 $ 139.00   0.72%      0
AAPL   2024-05-30   SELL  $ 144.00 $ 141.00  -2.08%      0

汇总:
  总交易笔数:   2
  匹配交易:     2
  仅回测有:     0
  仅实盘有:     0
  平均滑点:     -0.6793%
  手续费偏差:   $0


`ExecutionReport` 的核心价值：量化「理想回测」和「真实执行」之间的差距。如果滑点持续偏大，说明策略对执行价格敏感，需要优化下单时机或使用限价单。

**实战建议**：每日收盘后自动生成 ExecutionReport，监控滑点趋势。如果某段时间滑点突然增大，可能是市场流动性变差或策略容量已到上限。

---
## 5. Alpaca 集成 — 从模拟交易到实盘

> **注意**：本节需要 Alpaca Paper Trading API Key。如果未设置环境变量，代码会跳过实际调用，仅展示用法。

open-xquant 通过 `oxq.contrib.alpaca` 提供 Alpaca 集成：

| 组件 | 职责 |
|------|------|
| `AlpacaClient` | 底层 HTTP 客户端，封装认证和请求 |
| `AlpacaMarketDataProvider` | 行情数据，实现 `MarketDataProvider` 协议 |

### 完整工作流

```
收盘后:
  1. AlpacaMarketDataProvider.get_bars() 获取最新行情
  2. Engine.setup() + step() 跑策略
  3. generate_orders() 生成交易计划
  4. 交易员审核
  5. AlpacaClient 下单
  6. ExecutionReport 对比偏差
```

In [8]:
import os

from oxq.contrib.alpaca import AlpacaClient, AlpacaMarketDataProvider

HAS_ALPACA_KEY = bool(os.environ.get("ALPACA_API_KEY"))

if HAS_ALPACA_KEY:
    # 创建 Alpaca Paper Trading 客户端
    client = AlpacaClient(
        api_key=os.environ["ALPACA_API_KEY"],
        secret_key=os.environ["ALPACA_SECRET_KEY"],
        paper=True,  # 模拟交易模式
    )

    # 获取账户信息
    account = client.get("/v2/account")
    print(f"账户状态: {account['status']}")
    print(f"购买力:   ${float(account['buying_power']):,.2f}")
    print(f"净值:     ${float(account['equity']):,.2f}")

    # 创建行情数据 Provider
    alpaca_market = AlpacaMarketDataProvider(
        api_key=os.environ["ALPACA_API_KEY"],
        secret_key=os.environ["ALPACA_SECRET_KEY"],
    )

    # 获取最近行情
    bars = alpaca_market.get_bars("AAPL", "2024-01-01", "2024-01-31")
    print(f"\nAAPL 行情: {len(bars)} bars")
    print(bars.tail(3))
else:
    print("未设置 ALPACA_API_KEY，跳过 Alpaca API 调用。")
    print()
    print("如需运行本节，请设置环境变量：")
    print("  export ALPACA_API_KEY='your-paper-api-key'")
    print("  export ALPACA_SECRET_KEY='your-paper-secret-key'")
    print()
    print("以下展示代码用法（不实际执行）：")
    print("""
    from oxq.contrib.alpaca import AlpacaClient, AlpacaMarketDataProvider

    client = AlpacaClient(api_key="...", secret_key="...", paper=True)
    account = client.get("/v2/account")

    market = AlpacaMarketDataProvider(api_key="...", secret_key="...")
    bars = market.get_bars("AAPL", "2024-01-01", "2024-12-31")

    # 用真实行情跑策略
    engine = Engine()
    result = engine.run(strategy, market=market, broker=SimBroker(),
                        start="2024-01-01", end="2024-12-31")
    """)

未设置 ALPACA_API_KEY，跳过 Alpaca API 调用。

如需运行本节，请设置环境变量：
  export ALPACA_API_KEY='your-paper-api-key'
  export ALPACA_SECRET_KEY='your-paper-secret-key'

以下展示代码用法（不实际执行）：

    from oxq.contrib.alpaca import AlpacaClient, AlpacaMarketDataProvider

    client = AlpacaClient(api_key="...", secret_key="...", paper=True)
    account = client.get("/v2/account")

    market = AlpacaMarketDataProvider(api_key="...", secret_key="...")
    bars = market.get_bars("AAPL", "2024-01-01", "2024-12-31")

    # 用真实行情跑策略
    engine = Engine()
    result = engine.run(strategy, market=market, broker=SimBroker(),
                        start="2024-01-01", end="2024-12-31")
    


---
## 总结

本教程覆盖了 open-xquant 从回测到模拟交易的完整工作流：

| 功能 | 模块 | 核心 API |
|------|------|----------|
| 成交价压力测试 | `oxq.trade` | `FillPriceMode`, `SimBroker(fill_price_mode=...)` |
| 交易计划生成 | `oxq.trade` | `generate_orders()` → `PlannedOrder` |
| 逐 bar 执行 | `oxq.core` | `Engine.setup()` + `Engine.step(date)` |
| 成交偏差分析 | `oxq.portfolio` | `ExecutionReport(sim_fills, live_fills)` |
| Alpaca 集成 | `oxq.contrib.alpaca` | `AlpacaClient`, `AlpacaMarketDataProvider` |

**推荐工作流**：

1. 用 `CLOSE` 模式快速验证策略逻辑
2. 用 `NEXT_OPEN` 模式评估真实可行性
3. 用 `NEXT_HIGH`/`NEXT_LOW` 做压力测试
4. 用 `generate_orders()` 生成交易计划，人工审核
5. 用 `Engine.setup()` + `step()` 逐 bar 执行（实盘模式）
6. 用 `ExecutionReport` 每日监控滑点趋势